<div>
<img src="img/Pandas_logo.png" width="500"/>
</div>

# Pandas avancé — Opérations relationnelles et agrégations

Dans ce notebook, nous allons explorer des opérations très proches de celles que l'on rencontre
en bases de données relationnelles :

- **projection** et **restriction** ;
- **union** avec `pd.concat` ;
- **jointures** avec `pd.merge` ;
- différentes **cardinalités** de jointure ;
- **agrégations** avec `groupby`.

Une seconde partie propose un exercice complet sur des données mensuelles de consommation électrique.

## Exercice 1 — Opérations relationnelles avec Pandas

### 1.1 Projection et restriction

Nous commençons par un petit jeu de données décrivant plusieurs régions et leur production renouvelable.

- Une **projection** consiste à sélectionner certaines colonnes.
- Une **restriction** consiste à sélectionner certaines lignes.

In [1]:
import pandas as pd

regions = pd.read_csv("regions_energie.csv", sep=";")
regions.head()

,region,population_millions,production_renouvelable_gwh
0,Nord,6.1,8200
1,Ouest,4.8,10400
2,Centre,3.5,6900
3,Sud,5.7,12100


In [2]:
regions.shape

(4, 3)

In [3]:
print(regions)

   region  population_millions  production_renouvelable_gwh
0    Nord                  6.1                         8200
1   Ouest                  4.8                        10400
2  Centre                  3.5                         6900
3     Sud                  5.7                        12100


In [4]:
# Projection : une seule colonne
regions["production_renouvelable_gwh"]

0     8200
1    10400
2     6900
3    12100
Name: production_renouvelable_gwh, dtype: int64

In [5]:
# Restriction : ligne située à la position 1
regions.iloc[1]

region                         Ouest
population_millions              4.8
production_renouvelable_gwh    10400
Name: 1, dtype: object

<span style="color:blue">**QUESTION : Faire une projection sur les colonnes `region` et `population_millions`.**</span>

<span style="color:red">**CORRECTION :** Une projection sur plusieurs colonnes utilise une liste de noms de colonnes.</span>

In [6]:
regions[["region", "population_millions"]]

,region,population_millions
0,Nord,6.1
1,Ouest,4.8
2,Centre,3.5
3,Sud,5.7


<span style="color:blue">**QUESTION : Faire une restriction pour ne conserver que les régions produisant plus de 9 000 GWh d'énergie renouvelable.**</span>

<span style="color:red">**CORRECTION :** On crée un masque booléen sur la colonne puis on filtre les lignes.</span>

In [7]:
regions[regions["production_renouvelable_gwh"] > 9000]

,region,population_millions,production_renouvelable_gwh
1,Ouest,4.8,10400
3,Sud,5.7,12100


### 1.2 Union avec `pd.concat`

Unir deux tableaux revient à créer un troisième tableau contenant les lignes des deux premiers.
Par défaut, `pd.concat` les assemble verticalement.

In [8]:
regions2 = pd.read_csv("regions_energie2.csv", sep=";")
regions2

,region,population_millions,production_renouvelable_gwh
0,Est,5.2,9100
1,Sud-Ouest,4.1,8800
2,Littoral,2.9,7600


In [9]:
toutes_regions = pd.concat([regions, regions2])
toutes_regions

,region,population_millions,production_renouvelable_gwh
0,Nord,6.1,8200
1,Ouest,4.8,10400
2,Centre,3.5,6900
3,Sud,5.7,12100
0,Est,5.2,9100
1,Sud-Ouest,4.1,8800
2,Littoral,2.9,7600


<span style="color:blue">**QUESTION : Que remarquez-vous concernant les indices du résultat ?**</span>

<span style="color:red">**CORRECTION :** Les index d'origine sont conservés, donc certaines valeurs d'index apparaissent plusieurs fois. `ignore_index=True` permet de recréer un index continu.</span>

In [10]:
pd.concat([regions, regions2], ignore_index=True)

,region,population_millions,production_renouvelable_gwh
0,Nord,6.1,8200
1,Ouest,4.8,10400
2,Centre,3.5,6900
3,Sud,5.7,12100
4,Est,5.2,9100
5,Sud-Ouest,4.1,8800
6,Littoral,2.9,7600


### 1.3 Jointures avec `pd.merge`

Une jointure combine les informations de deux DataFrames à partir d'une ou plusieurs colonnes communes.

Nous allons utiliser un exemple de **formations en entreprise**.

#### 1.3.1 Cardinalité un-à-un

In [11]:
participants = pd.DataFrame({
    "participant": ["Amira", "Bastien", "Chloé", "Diego"],
    "service": ["Finance", "Data", "Data", "RH"]
})
participants

,participant,service
0,Amira,Finance
1,Bastien,Data
2,Chloé,Data
3,Diego,RH


In [12]:
inscriptions = pd.DataFrame({
    "participant": ["Chloé", "Amira", "Diego", "Bastien"],
    "annee_inscription": [2025, 2024, 2026, 2025]
})
inscriptions

,participant,annee_inscription
0,Chloé,2025
1,Amira,2024
2,Diego,2026
3,Bastien,2025


In [13]:
participants_inscrits = pd.merge(participants, inscriptions)
participants_inscrits

,participant,service,annee_inscription
0,Amira,Finance,2024
1,Bastien,Data,2025
2,Chloé,Data,2025
3,Diego,RH,2026


`pd.merge` détecte ici automatiquement la colonne commune `participant`.
Dans ce cas, chaque participant apparaît une seule fois dans chacun des deux tableaux :
la cardinalité est **un-à-un**.

<span style="color:blue">**QUESTION : Quelle colonne a servi de clé de jointure ?**</span>

<span style="color:red">**CORRECTION :** La colonne `participant`, présente dans les deux DataFrames.</span>

In [14]:
print(pd.merge(participants, inscriptions, on="participant"))

  participant  service  annee_inscription
0       Amira  Finance               2024
1     Bastien     Data               2025
2       Chloé     Data               2025
3       Diego       RH               2026


#### 1.3.2 Cardinalité plusieurs-à-un

In [15]:
responsables = pd.DataFrame({
    "service": ["Finance", "Data", "RH"],
    "responsable": ["Nora", "Yanis", "Sofia"]
})

pd.merge(participants_inscrits, responsables)

,participant,service,annee_inscription,responsable
0,Amira,Finance,2024,Nora
1,Bastien,Data,2025,Yanis
2,Chloé,Data,2025,Yanis
3,Diego,RH,2026,Sofia


<span style="color:blue">**QUESTION : Que remarquez-vous dans la colonne `responsable` ?**</span>

<span style="color:red">**CORRECTION :** Le responsable du service Data apparaît plusieurs fois, car plusieurs participants appartiennent au même service.</span>

In [16]:
pd.merge(participants_inscrits, responsables)

,participant,service,annee_inscription,responsable
0,Amira,Finance,2024,Nora
1,Bastien,Data,2025,Yanis
2,Chloé,Data,2025,Yanis
3,Diego,RH,2026,Sofia


#### 1.3.3 Cardinalité plusieurs-à-plusieurs

Chaque service peut proposer plusieurs modules de formation, et un même module peut être utile à plusieurs services.
Après jointure, un participant peut donc apparaître sur plusieurs lignes.

In [17]:
modules = pd.DataFrame({
    "service": ["Finance", "Finance", "Data", "Data", "RH", "RH"],
    "module": ["Excel avancé", "Budget", "Python", "SQL", "Recrutement", "Droit social"]
})

plan_formation = pd.merge(participants_inscrits, modules)
plan_formation

,participant,service,annee_inscription,module
0,Amira,Finance,2024,Excel avancé
1,Amira,Finance,2024,Budget
2,Bastien,Data,2025,Python
3,Bastien,Data,2025,SQL
4,Chloé,Data,2025,Python
5,Chloé,Data,2025,SQL
6,Diego,RH,2026,Recrutement
7,Diego,RH,2026,Droit social


<span style="color:blue">**QUESTION : Comment interpréter les nouvelles lignes créées par la jointure ?**</span>

<span style="color:red">**CORRECTION :** Chaque ligne représente maintenant une combinaison participant–module. Un participant apparaît donc autant de fois qu'il existe de modules associés à son service.</span>

In [18]:
plan_formation

,participant,service,annee_inscription,module
0,Amira,Finance,2024,Excel avancé
1,Amira,Finance,2024,Budget
2,Bastien,Data,2025,Python
3,Bastien,Data,2025,SQL
4,Chloé,Data,2025,Python
5,Chloé,Data,2025,SQL
6,Diego,RH,2026,Recrutement
7,Diego,RH,2026,Droit social


### 1.4 Agrégation avec `groupby`

Comme le `GROUP BY` de SQL, `groupby` permet de partitionner les lignes selon une ou plusieurs colonnes,
puis d'appliquer une fonction d'agrégation.

In [19]:
resultats = pd.DataFrame({
    "groupe": ["A", "B", "C", "A", "B", "C"],
    "score": [14, 11, 16, 18, 13, 15],
    "duree_min": [45, 52, 48, 39, 50, 46]
})
print(resultats)

  groupe  score  duree_min
0      A     14         45
1      B     11         52
2      C     16         48
3      A     18         39
4      B     13         50
5      C     15         46


In [20]:
gb = resultats.groupby("groupe")
print("Sommes :")
print(gb.sum())

print("\nMoyennes :")
print(gb.mean())

Sommes :
        score  duree_min
groupe                  
A          32         84
B          24        102
C          31         94

Moyennes :
        score  duree_min
groupe                  
A        16.0       42.0
B        12.0       51.0
C        15.5       47.0


<span style="color:blue">**QUESTION : Calculer uniquement le score moyen de chaque groupe.**</span>

<span style="color:red">**CORRECTION :** On sélectionne la colonne `score` après le `groupby`, puis on applique `mean()`.</span>

In [21]:
resultats.groupby("groupe")["score"].mean()

groupe
A    16.0
B    12.0
C    15.5
Name: score, dtype: float64

## Exercice 2 — Analyse de consommation électrique mensuelle

Le fichier `consommation_electrique.csv` contient, pour chaque mois entre 2019 et 2025 :

- le mois et l'année ;
- la consommation électrique en GWh ;
- la température moyenne en °C ;
- la part d'électricité renouvelable en pourcentage.

Le jeu de données est **synthétique**, afin de permettre un exercice reproductible.

In [22]:
df = pd.read_csv("consommation_electrique.csv", sep=";")
df.head()

,Mois,Annee,Consommation_GWh,Temperature_C,PartRenouvelable_pct
0,Janvier,2019,122.0,4.6,36.6
1,Fevrier,2019,118.6,7.1,37.7
2,Mars,2019,103.1,9.7,37.7
3,Avril,2019,91.0,12.1,35.9
4,Mai,2019,85.6,15.2,40.3


<span style="color:blue">**QUESTION : Trouver le nombre de lignes et de colonnes de cette table avec `shape`.**</span>

<span style="color:red">**CORRECTION :** Le tuple retourné correspond au nombre de lignes puis au nombre de colonnes.</span>

In [23]:
df.shape

(84, 5)

<span style="color:blue">**QUESTION : Lister les noms des colonnes avec `columns`.**</span>

<span style="color:red">**CORRECTION :** `df.columns` renvoie l'index contenant les noms de colonnes.</span>

In [24]:
print(df.columns)

Index(['Mois', 'Annee', 'Consommation_GWh', 'Temperature_C',
       'PartRenouvelable_pct'],
      dtype='object')


<span style="color:blue">**QUESTION : Afficher les types des données de chaque colonne avec `dtypes`.**</span>

<span style="color:red">**CORRECTION :** `dtypes` permet de vérifier les types utilisés par Pandas.</span>

In [25]:
print(df.dtypes)

Mois                     object
Annee                     int64
Consommation_GWh        float64
Temperature_C           float64
PartRenouvelable_pct    float64
dtype: object


<span style="color:blue">**QUESTION : Afficher des informations générales sur chaque colonne avec `info()`.**</span>

<span style="color:red">**CORRECTION :** `info()` résume notamment le nombre de valeurs non nulles, les types et la mémoire utilisée.</span>

In [26]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 84 entries, 0 to 83
Data columns (total 5 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Mois                  84 non-null     object 
 1   Annee                 84 non-null     int64  
 2   Consommation_GWh      84 non-null     float64
 3   Temperature_C         84 non-null     float64
 4   PartRenouvelable_pct  84 non-null     float64
dtypes: float64(3), int64(1), object(1)
memory usage: 3.4+ KB


<span style="color:blue">**QUESTION : Afficher la colonne `Mois`.**</span>

<span style="color:red">**CORRECTION :** On peut sélectionner une colonne avec des crochets.</span>

In [27]:
print(df["Mois"])

0       Janvier
1       Fevrier
2          Mars
3         Avril
4           Mai
        ...    
79         Août
80    Septembre
81      Octobre
82     Novembre
83     Décembre
Name: Mois, Length: 84, dtype: object


<span style="color:blue">**QUESTION : Afficher simultanément les colonnes `Mois` et `Annee`.**</span>

<span style="color:red">**CORRECTION :** Pour sélectionner plusieurs colonnes, on transmet une liste de noms.</span>

In [28]:
print(df[["Mois", "Annee"]])

         Mois  Annee
0     Janvier   2019
1     Fevrier   2019
2        Mars   2019
3       Avril   2019
4         Mai   2019
..        ...    ...
79       Août   2025
80  Septembre   2025
81    Octobre   2025
82   Novembre   2025
83   Décembre   2025

[84 rows x 2 columns]


<span style="color:blue">**QUESTION : Lister les mois uniques et compter leur nombre.**</span>

<span style="color:red">**CORRECTION :** `unique()` renvoie les valeurs distinctes ; `len()` permet de les compter.</span>

In [29]:
mois_uniques = df["Mois"].unique()
print(mois_uniques)
print("Nombre :", len(mois_uniques))

['Janvier' 'Fevrier' 'Mars' 'Avril' 'Mai' 'Juin' 'Juillet' 'Août'
 'Septembre' 'Octobre' 'Novembre' 'Décembre']
Nombre : 12


Vous devriez remarquer une incohérence typographique : `Fevrier` ne comporte pas d'accent,
alors que les autres noms de mois utilisent les accents français.

<span style="color:blue">**QUESTION : Corriger `Fevrier` en `Février` *en place* et en une seule instruction.**</span>

<span style="color:red">**CORRECTION :** `replace` peut remplacer une valeur dans tout le DataFrame et `inplace=True` modifie directement l'objet.</span>

In [30]:
df.replace("Fevrier", "Février", inplace=True)

<span style="color:blue">**QUESTION : Afficher les statistiques descriptives de la colonne `Consommation_GWh`.**</span>

<span style="color:red">**CORRECTION :** `describe()` fournit notamment le nombre de valeurs, la moyenne, l'écart-type, les quartiles, le minimum et le maximum.</span>

In [31]:
print(df["Consommation_GWh"].describe())

count     84.000000
mean     102.028571
std       19.349092
min       71.100000
25%       85.175000
50%      100.850000
75%      118.675000
max      135.400000
Name: Consommation_GWh, dtype: float64


<span style="color:blue">**QUESTION : Lister les périodes après 2022 pour lesquelles la consommation est supérieure ou égale à 115 GWh.**</span>

<span style="color:red">**CORRECTION :** On combine deux conditions booléennes avec `&` et on sélectionne les colonnes `Mois` et `Annee`.</span>

In [32]:
forte_conso = df.loc[(df["Consommation_GWh"] >= 115) & (df["Annee"] > 2022), :]
forte_conso[["Mois", "Annee"]]

,Mois,Annee
48,Janvier,2023
49,Février,2023
58,Novembre,2023
59,Décembre,2023
60,Janvier,2024
61,Février,2024
62,Mars,2024
70,Novembre,2024
71,Décembre,2024
72,Janvier,2025


<span style="color:blue">**QUESTION : Dans cette sélection, compter le nombre de mois par année.**</span>

<span style="color:red">**CORRECTION :** `value_counts()` appliqué à `Annee` compte le nombre de lignes pour chaque année.</span>

In [33]:
forte_conso["Annee"].value_counts().sort_index()

2023    4
2024    5
2025    5
Name: Annee, dtype: int64

<span style="color:blue">**QUESTION : Dans cette même sélection, compter combien de fois apparaît chaque mois, toutes années confondues.**</span>

<span style="color:red">**CORRECTION :** `value_counts()` sur la colonne `Mois` donne le nombre d'occurrences de chaque mois.</span>

In [34]:
forte_conso["Mois"].value_counts()

Janvier     3
Février     3
Novembre    3
Décembre    3
Mars        2
Name: Mois, dtype: int64

<span style="color:blue">**QUESTION : À partir du tableau initial, afficher toutes les observations du mois de janvier.**</span>

<span style="color:red">**CORRECTION :** On peut utiliser `groupby(...).get_group(...)`, comme dans le support d'origine.</span>

In [35]:
dMois = df.groupby("Mois")
dJanvier = dMois.get_group("Janvier")
dJanvier

,Mois,Annee,Consommation_GWh,Temperature_C,PartRenouvelable_pct
0,Janvier,2019,122.0,4.6,36.6
12,Janvier,2020,129.4,7.1,36.7
24,Janvier,2021,131.5,6.8,38.8
36,Janvier,2022,133.0,7.1,38.9
48,Janvier,2023,133.4,6.2,37.1
60,Janvier,2024,132.5,5.1,37.9
72,Janvier,2025,134.7,5.4,35.5


<span style="color:blue">**QUESTION : Combien d'observations contient ce groupe ?**</span>

<span style="color:red">**CORRECTION :** Il y a une observation de janvier par année du jeu de données.</span>

In [36]:
len(dJanvier)

7

<span style="color:blue">**QUESTION : Calculer la consommation électrique moyenne pour chaque mois.**</span>

<span style="color:red">**CORRECTION :** On groupe par mois puis on calcule la moyenne de `Consommation_GWh`.</span>

In [37]:
conso_moyenne_mois = df.groupby("Mois")["Consommation_GWh"].mean()
conso_moyenne_mois

Mois
Août          78.371429
Avril         97.671429
Décembre     127.885714
Février      122.071429
Janvier      130.928571
Juillet       78.071429
Juin          80.342857
Mai           88.742857
Mars         111.328571
Novembre     117.228571
Octobre      102.414286
Septembre     89.285714
Name: Consommation_GWh, dtype: float64

<span style="color:blue">**QUESTION : Trier le tableau initial par consommation décroissante.**</span>

<span style="color:red">**CORRECTION :** `sort_values` trie les lignes selon la colonne choisie ; `ascending=False` donne un ordre décroissant.</span>

In [38]:
df_trie = df.sort_values(by="Consommation_GWh", ascending=False)
df_trie

,Mois,Annee,Consommation_GWh,Temperature_C,PartRenouvelable_pct
83,Décembre,2025,135.4,5.2,30.2
72,Janvier,2025,134.7,5.4,35.5
59,Décembre,2023,133.6,5.8,38.4
48,Janvier,2023,133.4,6.2,37.1
71,Décembre,2024,133.1,6.9,35.3
...,...,...,...,...,...
7,Août,2019,76.5,24.8,33.8
5,Juin,2019,76.4,21.8,35.2
18,Juillet,2020,72.4,22.3,33.0
19,Août,2020,71.3,23.3,34.2


<span style="color:blue">**QUESTION : Trouver la consommation maximale.**</span>

<span style="color:red">**CORRECTION :** La méthode `max()` renvoie la valeur maximale de la colonne.</span>

In [39]:
df["Consommation_GWh"].max()

135.4

<span style="color:blue">**QUESTION : À quel mois et à quelle année cette consommation maximale correspond-elle ?**</span>

<span style="color:red">**CORRECTION :** On filtre les lignes dont la consommation est égale au maximum, puis on affiche le mois et l'année.</span>

In [40]:
df_max = df[df["Consommation_GWh"] == df["Consommation_GWh"].max()]
df_max[["Mois", "Annee", "Consommation_GWh"]]

,Mois,Annee,Consommation_GWh
83,Décembre,2025,135.4


<span style="color:blue">**QUESTION : Comparer la température moyenne des mois où la consommation dépasse 115 GWh avec celle des autres mois.**</span>

<span style="color:red">**CORRECTION :** On peut construire un masque puis calculer deux moyennes. Le jeu synthétique a été conçu pour illustrer une consommation plus forte pendant les mois froids.</span>

In [41]:
masque = df["Consommation_GWh"] > 115
print("Température moyenne, forte conso :", df.loc[masque, "Temperature_C"].mean())
print("Température moyenne, autres mois :", df.loc[~masque, "Temperature_C"].mean())

Température moyenne, forte conso : 6.796296296296297
Température moyenne, autres mois : 16.763157894736842


## Exercice 3 — Petit exercice de synthèse

Utilisez maintenant les notions vues dans les deux parties précédentes.

<span style="color:blue">**QUESTION : Calculer, par année, la consommation moyenne et la part renouvelable moyenne.**</span>

<span style="color:red">**CORRECTION :** On peut grouper par année puis sélectionner les deux colonnes numériques à agréger.</span>

In [42]:
df.groupby("Annee")[["Consommation_GWh", "PartRenouvelable_pct"]].mean()

,Consommation_GWh,PartRenouvelable_pct
Annee,,
2019,96.183333,34.325000
2020,98.316667,34.800000
2021,100.558333,35.991667
2022,101.300000,36.608333
2023,104.233333,37.158333
2024,107.191667,37.675000
2025,106.416667,37.108333


<span style="color:blue">**QUESTION : Trouver l'année où la part renouvelable moyenne est la plus élevée.**</span>

<span style="color:red">**CORRECTION :** On calcule la moyenne annuelle puis `idxmax()` renvoie l'année correspondant au maximum.</span>

In [43]:
renouv_annuel = df.groupby("Annee")["PartRenouvelable_pct"].mean()
print(renouv_annuel.idxmax())

2024


<span style="color:blue">**QUESTION : Créer deux catégories de température : `froid` pour moins de 12 °C, `doux/chaud` sinon, puis calculer la consommation moyenne de chaque catégorie.**</span>

<span style="color:red">**CORRECTION :** On peut créer une colonne temporaire avec `np.where`, puis grouper sur cette nouvelle catégorie.</span>

In [44]:
import numpy as np
df2 = df.copy()
df2["categorie_temperature"] = np.where(df2["Temperature_C"] < 12, "froid", "doux/chaud")
df2.groupby("categorie_temperature")["Consommation_GWh"].mean()

categorie_temperature
doux/chaud     87.076087
froid         120.128947
Name: Consommation_GWh, dtype: float64

## Ressources

- Référence API Pandas : https://pandas.pydata.org/docs/reference/
- Guide utilisateur Pandas : https://pandas.pydata.org/docs/user_guide/

À retenir : les opérations `concat`, `merge` et `groupby` sont essentielles dès que l'on manipule
plusieurs tables ou que l'on souhaite produire des indicateurs par groupe.